# 03 — Exploratory Data Analysis
## PM2.5 & Weather Relationships — Cedar Creek Fire, Eugene OR 2022

Analyses in this notebook:
1. Full event time series (PM2.5 + weather panels)
2. Smoke vs non-smoke period comparison
3. Correlation heatmap
4. PM2.5 vs individual weather variables (scatter)
5. Diurnal (hour-of-day) patterns
6. Wind direction analysis

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Smoke threshold: EPA 24-hour standard
SMOKE_THRESHOLD = 35  # µg/m³

## 0. Load Processed Data

In [ ]:
df = pd.read_csv("../data/processed/analysis_data.csv", parse_dates=["timestamp"])
print(f"Loaded {len(df):,} records")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

# Flag smoke hours
df['is_smoke'] = df['pm2.5_lrapa'] >= SMOKE_THRESHOLD
print(f"\nSmoke hours (≥ {SMOKE_THRESHOLD} µg/m³): {df['is_smoke'].sum()} ({df['is_smoke'].mean()*100:.1f}%)")

## 1. The Cedar Creek Fire Smoke Event — Full Time Series

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

panels = [
    ('pm2.5_lrapa',           'PM2.5 LRAPA-corrected (µg/m³)', 'tomato'),
    ('pm2.5_lrapa_regulatory','PM2.5 Regulatory LRAPA (µg/m³)', 'black'),
    ('temperature_f',         'Temperature (°F)', '#e67e22'),
    ('humidity',              'Relative Humidity (%)', 'steelblue'),
    ('wind_speed_mph',        'Wind Speed (mph)', 'seagreen'),
]

for ax, (col, label, color) in zip(axes, panels):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    ax.fill_between(df['timestamp'], df[col], alpha=0.25, color=color)
    ax.plot(df['timestamp'], df[col], color=color, linewidth=0.9)
    ax.set_ylabel(label, fontsize=10)
    if 'pm2.5' in col:
        ax.axhline(SMOKE_THRESHOLD, color='orange', linestyle='--', linewidth=1, alpha=0.8)

# Shade peak smoke period (approx Sep 4–12)
for ax in axes:
    ax.axvspan(pd.Timestamp('2022-09-04'), pd.Timestamp('2022-09-12'),
               alpha=0.08, color='red')

smoke_patch = mpatches.Patch(color='red', alpha=0.2, label='Peak smoke period')
axes[0].legend(handles=[smoke_patch], loc='upper right', fontsize=9)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
axes[0].set_title('Cedar Creek Fire Smoke Event — Eugene OR (Aug–Sep 2022)', fontsize=13)
plt.tight_layout()
plt.savefig('../data/processed/fig_full_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Smoke vs Non-Smoke Period Comparison

In [ ]:
weather_cols = ['temperature_f', 'humidity', 'wind_speed_mph', 'pressure_hpa']
weather_cols = [c for c in weather_cols if c in df.columns]

smoke_df     = df[df['is_smoke']]
no_smoke_df  = df[~df['is_smoke']]

print(f"Smoke hours (≥{SMOKE_THRESHOLD} µg/m³): n = {len(smoke_df)}")
print(f"Non-smoke hours:                       n = {len(no_smoke_df)}")
print()

fig, axes = plt.subplots(1, len(weather_cols), figsize=(14, 5))
for ax, col in zip(axes, weather_cols):
    data_smoke    = smoke_df[col].dropna()
    data_no_smoke = no_smoke_df[col].dropna()
    ax.boxplot([data_no_smoke, data_smoke],
               labels=['No smoke', 'Smoke'],
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    t, p = stats.ttest_ind(data_smoke, data_no_smoke, equal_var=False)
    ax.set_title(f'{col}\np = {p:.3f}', fontsize=10)
    ax.set_ylabel(col)

plt.suptitle('Weather Conditions: Smoke vs Non-Smoke Hours', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/fig_smoke_vs_nonsmoke.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Correlation Heatmap

In [ ]:
corr_cols = ['pm2.5_lrapa', 'pm2.5_lrapa_regulatory',
             'temperature_f', 'humidity', 'wind_speed_mph',
             'wind_direction', 'pressure_hpa', 'precipitation_in']
corr_cols = [c for c in corr_cols if c in df.columns]

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Pearson Correlation Matrix — PM2.5 & Weather Variables')
plt.tight_layout()
plt.savefig('../data/processed/fig_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Print PM2.5 correlations sorted
print("Correlations with pm2.5_lrapa:")
print(corr['pm2.5_lrapa'].drop('pm2.5_lrapa').sort_values().to_string())

## 4. PM2.5 vs Individual Weather Variables

In [ ]:
scatter_cols = [c for c in ['temperature_f','humidity','wind_speed_mph','pressure_hpa'] if c in df.columns]
fig, axes = plt.subplots(1, len(scatter_cols), figsize=(14, 4))

for ax, col in zip(axes, scatter_cols):
    clean = df[['pm2.5_lrapa', col]].dropna()
    r, p  = stats.pearsonr(clean['pm2.5_lrapa'], clean[col])
    ax.scatter(clean[col], clean['pm2.5_lrapa'],
               s=5, alpha=0.3, color='steelblue')
    # Trend line
    m, b = np.polyfit(clean[col], clean['pm2.5_lrapa'], 1)
    x = np.linspace(clean[col].min(), clean[col].max(), 100)
    ax.plot(x, m*x + b, 'r-', linewidth=1.5)
    ax.set_xlabel(col)
    ax.set_ylabel('PM2.5 LRAPA (µg/m³)' if ax == axes[0] else '')
    ax.set_title(f'r = {r:.2f}, p = {p:.3f}', fontsize=10)

plt.suptitle('PM2.5 vs Weather Variables', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/fig_pm25_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Diurnal Patterns (Hour of Day)

In [ ]:
diurnal_cols = [c for c in ['pm2.5_lrapa','temperature_f','humidity','wind_speed_mph'] if c in df.columns]
fig, axes = plt.subplots(1, len(diurnal_cols), figsize=(14, 4))

for ax, col in zip(axes, diurnal_cols):
    hourly = df.groupby('hour')[col].agg(['mean','sem']).reset_index()
    ax.fill_between(hourly['hour'],
                    hourly['mean'] - hourly['sem'],
                    hourly['mean'] + hourly['sem'],
                    alpha=0.25, color='steelblue')
    ax.plot(hourly['hour'], hourly['mean'], color='steelblue', linewidth=2)
    ax.set_xlabel('Hour of day (Pacific)')
    ax.set_ylabel(col)
    ax.set_xticks(range(0, 24, 4))

plt.suptitle('Diurnal Patterns — Mean ± SE by Hour (Aug–Sep 2022)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/fig_diurnal.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Wind Direction Analysis

In [ ]:
if 'wind_direction' in df.columns:
    wd = df[['wind_direction', 'pm2.5_lrapa', 'wind_speed_mph']].dropna()
    # Bin wind direction into 16 compass sectors
    bins = np.arange(-11.25, 371.25, 22.5)
    labels = range(16)
    wd['sector'] = pd.cut(wd['wind_direction'] % 360, bins=bins, labels=labels)
    wd_mean = wd.groupby('sector')['pm2.5_lrapa'].mean()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Bar chart: mean PM2.5 by wind direction sector
    ax = axes[0]
    sector_angles = np.arange(0, 360, 22.5)
    compass = ['N','NNE','NE','ENE','E','ESE','SE','SSE',
               'S','SSW','SW','WSW','W','WNW','NW','NNW']
    bars = ax.bar(range(16), wd_mean, color=plt.cm.YlOrRd(wd_mean / wd_mean.max()))
    ax.set_xticks(range(16))
    ax.set_xticklabels(compass, rotation=45, fontsize=8)
    ax.set_ylabel('Mean PM2.5 LRAPA (µg/m³)')
    ax.set_title('Mean PM2.5 by Wind Direction')

    # Scatter: wind direction vs PM2.5
    ax = axes[1]
    ax.scatter(wd['wind_direction'], wd['pm2.5_lrapa'],
               s=4, alpha=0.3, color='steelblue')
    ax.set_xlabel('Wind Direction (degrees)')
    ax.set_ylabel('PM2.5 LRAPA (µg/m³)')
    ax.set_title('PM2.5 vs Wind Direction')
    ax.set_xticks([0, 90, 180, 270, 360])
    ax.set_xticklabels(['N (0°)', 'E (90°)', 'S (180°)', 'W (270°)', 'N (360°)'])

    plt.tight_layout()
    plt.savefig('../data/processed/fig_wind_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()